# Stage I. Operating threshold and test-pool evaluation

**Steps.**
1. Load the best checkpoint with its EMA weights.
2. Fix the operating threshold on validation data.
3. Validation predictions with test-time augmentation (TTA).
4. Test-pool predictions with TTA (cached, resumable).
5. Test metrics at the frozen threshold, routing by true grade, calibration.
6. Figure: ROC and routing rate per grade.
7. Save the per-image predictions and the report.

TTA averages four views: identity, horizontal flip, vertical flip and one random rotation drawn
from [-8, +8] degrees. Images are processed in fixed chunks (500 validation, 1,000 test) and the
random generators are re-seeded with 42 + chunk index, so the rotation views are repeatable.

## 0. Configuration

In [ ]:
from pathlib import Path

# All paths are relative to the repository root (start Jupyter there, or edit them here).
#   data/train/train_grade.csv     development pool labels: image, grade (0-4; 5 = ungradable)
#   data/train/train_image/        development pool images (51,445 in the paper)
#   data/test/test_grade.csv       test pool labels: image, grade (58,689 images)
#   data/test/test_image/          test pool images
DATA_ROOT = Path("data")
TRAIN_CSV = DATA_ROOT / "train" / "train_grade.csv"
TRAIN_IMG_DIR = DATA_ROOT / "train" / "train_image"
TEST_CSV = DATA_ROOT / "test" / "test_grade.csv"
TEST_IMG_DIR = DATA_ROOT / "test" / "test_image"

OUT_ROOT = Path("outputs")
RUN_DIR = OUT_ROOT / "stage1" / "convnextv2_768"     # checkpoints, history, split
EVAL_DIR = OUT_ROOT / "stage1" / "eval"              # prediction caches and reports

BEST_CKPT = RUN_DIR / "best_auc_model.pth"
SPLIT_CSV = RUN_DIR / "split.csv"                    # written by notebook 01
TEST_CACHE = EVAL_DIR / "test_preds_768.csv"         # key (relative path), prob
VAL_CACHE = EVAL_DIR / "val_preds_768.csv"
RUN_VALIDATION_TTA = True
THRESHOLD_SOURCE = "checkpoint"   # "checkpoint" (paper) or "tta_val" (same rule on TTA preds)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import json
import numpy as np
import pandas as pd
import torch
from hierarchiretina.stage1 import data, engine
from hierarchiretina.stage1 import evaluate as ev
from hierarchiretina.stage1 import model as M
from hierarchiretina.stage1.config import GATE_THRESHOLD, get_preset
from hierarchiretina.stage1.preprocessing import get_preprocess

cfg = get_preset("convnextv2_768")
DEVICE = engine.get_device()
PRE = get_preprocess(cfg.preprocess)
TTA = data.build_tta_transforms(cfg.tta_rotate_limit)

## 1. Model

In [ ]:
model = M.build_model(cfg, pretrained=False)
meta = M.load_eval_weights(model, BEST_CKPT)
model.to(DEVICE)
if cfg.channels_last:
    model.to(memory_format=torch.channels_last)
SIG = ev.checkpoint_signature(BEST_CKPT) + f"|{cfg.img_size}"
predict = lambda paths: ev.tta_predict_paths(model, paths, PRE, cfg.img_size, TTA, DEVICE,
                                             cfg.amp, cfg.channels_last, img_batch=4)
pd.Series({k: meta.get(k) for k in ("epoch", "auc", "grade1_acc", "threshold")})

## 2. Operating threshold

The threshold maximises 2 TPR - FPR (a Youden index weighted towards sensitivity) on the
validation predictions of the selected epoch, i.e. the single-view EMA pass made during
training, clipped to [0.20, 0.60]. It is stored in the checkpoint and frozen before the test pool
is scored: tau = 0.2391715943813324. The same rule on the TTA validation predictions of step 3
gives 0.2485 and is reported for reference only.

In [ ]:
tau = float(meta["threshold"])
print(f"checkpoint threshold {tau!r} | paper {GATE_THRESHOLD!r} | equal: {tau == GATE_THRESHOLD}")

## 3. Validation predictions (TTA)

In [ ]:
if RUN_VALIDATION_TTA:
    split = pd.read_csv(SPLIT_CSV)
    val_df = split[split.split == "val"].reset_index(drop=True)
    val_probs = ev.run_chunked_tta(predict, TTA, val_df.img_path.tolist(),
                                   val_df.img_path.tolist(), VAL_CACHE, chunk_size=500,
                                   seed=cfg.seed, signature=SIG)
    tau_tta = ev.find_optimal_threshold(val_df.binary_label, val_probs, "2tpr-fpr", (0.2, 0.6))
    if THRESHOLD_SOURCE == "tta_val":
        tau = tau_tta
    val_auc = ev.compute_metrics(val_df.binary_label, val_probs)["auc"]
    print(f"validation AUC (TTA + EMA) {val_auc:.4f} | TTA-val threshold {tau_tta:.4f}")
    display(ev.per_grade_routing(val_df.grade, val_probs, tau))
(EVAL_DIR / "eval_threshold_768.json").write_text(json.dumps(
    {"threshold": tau, "source": THRESHOLD_SOURCE, "ckpt_epoch": meta.get("epoch")}, indent=2))

## 4. Test-pool predictions (TTA)

58,689 images, sorted by path; the cache key is the path relative to `TEST_IMG_DIR`. A cache
copied from the original run (`test_preds_768.csv`, columns `key, prob`) is used as is.

In [ ]:
test_paths = data.list_images(TEST_IMG_DIR)
keys = [p.relative_to(TEST_IMG_DIR).as_posix() for p in test_paths]
test_probs = ev.run_chunked_tta(predict, TTA, test_paths, keys, TEST_CACHE, chunk_size=1000,
                                seed=cfg.seed, signature=SIG)
print(f"{len(test_probs):,} test images scored")

## 5. Test metrics at the frozen threshold

Expected (paper): AUC 0.929 (95 % CI 0.926-0.932, 500 bootstrap resamples), sensitivity 0.854,
specificity 0.887, PPV 0.752, NPV 0.938; 19,154 images routed to Stage II and 39,535 stopped
as Grade 0; sensitivity 0.918 for Grades 2-4 and 0.978 for Grades 3-4; 345 of 346 ungradable
images passed; 4,745 healthy eyes admitted; ECE 0.041 (15 bins) and Brier score 0.078.

In [ ]:
gt = data.load_test_labels(TEST_CSV)
pf = pd.DataFrame({"stem": [p.stem for p in test_paths], "prob": test_probs})
mg = gt.merge(pf, on="stem", how="inner")
y, p, g = mg.gt_binary.values, mg.prob.values, mg.true_grade.values
summary = ev.screening_summary(y, p, g, tau)
summary["auc_ci95"] = ev.bootstrap_ci(y, p, n_boot=500, seed=cfg.seed)
pd.Series(summary).to_frame("value")

In [ ]:
ev.per_grade_routing(g, p, tau)

Threshold sweep on the test pool, for reporting only (the threshold is never tuned on test
data).

In [ ]:
ev.threshold_sweep(y, p, g, [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]).round(4)

## 6. Figure: ROC with the operating point and routing by true grade

In [ ]:
fig = ev.plot_gate_figure(y, p, g, tau, EVAL_DIR / "fig_stage1.png")

## 7. Save

In [ ]:
mg.assign(pred=(p >= tau).astype(int)).rename(
    columns={"true_grade": "grade", "gt_binary": "gt"})[["stem", "grade", "gt", "prob", "pred"]
].to_csv(EVAL_DIR / "test_predictions_768.csv", index=False)
pd.DataFrame([{k: (str(v) if isinstance(v, tuple) else v) for k, v in summary.items()}]).to_csv(
    EVAL_DIR / "test_report_768.csv", index=False)
print("saved:", EVAL_DIR / "test_predictions_768.csv", EVAL_DIR / "test_report_768.csv")